# Introduction to HPC Clusters and SLURM

In high‑performance computing (HPC) a **cluster** is a collection of compute nodes that work together to solve large problems.  This notebook provides a concise, hands‑on overview of the key components of a cluster, the SLURM job manager, and basic commands you can run directly from the notebook.

## What is a Cluster?

A typical HPC cluster consists of:
- **Head (login) node** – where you log in, edit scripts and submit jobs.
- **Compute nodes** – the machines that actually run your parallel programs.
- **Network interconnect** – high‑speed fabric (InfiniBand, Ethernet) that lets the nodes talk to each other.
- **Shared storage** – a parallel filesystem (e.g., Lustre, GPFS) visible from every node.
- **Scheduler / resource manager** – software that decides *when* and *where* a job runs.  In most modern clusters this scheduler is **SLURM** (Simple Linux Utility for Resource Management).

![Cluster Overview](./img/cluster_overview.png)

## SLURM – the Job Manager

SLURM has three core daemons:
- **slurmctld** – the central controller (runs on the head node).
- **slurmd** – a daemon on each compute node that launches the tasks.
- **slurmdbd** – optional database daemon for accounting.

Resources are organised into **partitions** (think of them as queues).  When you submit a job you specify the partition, the number of nodes, the number of CPU‑cores per task, etc.  SLURM matches your request against available resources and starts the job when the resources become free.

![SLURM Architecture](./img/slurm_arch.png)

## Basic SLURM Commands (run them with the `!` prefix)

These commands work on any system where SLURM is installed.  If you run them on a local machine without SLURM they will return an error – that is expected.

In [ ]:
!sinfo  # Show available partitions and their state

In [ ]:
!squeue -u $USER  # List your jobs in the queue

## Submitting a Simple SLURM Job

We will create a tiny Bash script that prints the node name and then sleeps for a few seconds.  The script is written to a temporary file, submitted with `sbatch`, and then we query its status.

In [ ]:
script = '''#!/bin/bash
#SBATCH --job-name=example_job
#SBATCH --output=example_job.out
#SBATCH --error=example_job.err
#SBATCH --time=00:01:00
#SBATCH --ntasks=2

echo "Running on host $(hostname)"
sleep 10
'''
with open('example_job.sh', 'w') as f:
    f.write(script)
!chmod +x example_job.sh

In [ ]:
!sbatch example_job.sh  # Submit the job

In [ ]:
!squeue -u $USER  # Check the job in the queue (replace <jobid> with the ID printed by sbatch)

## Monitoring a Running Job

* `scontrol show job <jobid>` – detailed information about a specific job.
* `sacct -j <jobid> -o JobID,State,Start,End,Elapsed,MaxRSS` – accounting data after the job finishes.
* `sstat -j <jobid> -o AveCPU,MaxRSS,MaxVMSize` – live statistics while the job is running.
* `sview` – optional GUI if you have X‑forwarding to the head node.

In [ ]:
!sacct -X -n -s RUNNING -o JobID%15,State%10,NodeList%20 -u $USER | tail -n1  # Show most recent running job

## Interactive Jobs with `srun`

If you just want to launch a command on a compute node without writing a script, use `srun`.  The following example starts four parallel tasks that each print the hostname of the node they run on.

In [ ]:
!srun -n 4 hostname

## Cancelling a Job

If you need to stop a job, use `scancel <jobid>`.  You can obtain the job id from `squeue` or `sacct`.

In [ ]:
# !scancel <jobid>  # Uncomment and replace <jobid> to cancel

![Network Fabric](./img/network_fabric.png)
---
*This notebook gives a quick hands‑on tour of clusters and SLURM.  For deeper details consult the official SLURM documentation or the `man` pages (`man sinfo`, `man sbatch`, etc.).*